# Makespan su macchine identiche (capitolo 3)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/modellazione-mip/blob/main/notebooks/cap06_cmax.ipynb)

Il secondo dei tre problemi che il capitolo delle euristiche riprende: li' si
confronta l'ordine naturale con LPT sugli stessi quattro lavori. L'obiettivo e'
una sola variabile, z, e sono i vincoli di carico a darle significato: e' la
tecnica min-max.

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/modellazione-mip/gurobipy-4/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica i quattro moduli comuni del
corso: `stile.py` (palette), `mip.py` (rilassamento, duale, bound),
`euristiche.py` (next-fit, first-fit, best-fit) ed `esteso.py` (il modello
dell'istanza scritto per esteso). La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: le istanze del corso sono piccole e ci stanno tutte
con ampio margine. Per istanze più grandi si attiva la licenza accademica
gratuita da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e i moduli comuni del corso.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche', 'esteso', 'booleane'):                     # stile grafico e utilità del corso
    locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))       # notebook aperto nel repository
    else:                                 # su Colab si riscarica sempre: la sessione dura,
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/modellazione-mip/main/python/{modulo}.py", f"{modulo}.py")   # i moduli cambiano

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from esteso import salva_modello
from mip import frazione, nuovo_modello, risolvi
from stile import intestazione, salva_dati

R = range

intestazione("Makespan: il carico della macchina piu' carica")
d_cmax = [3, 4, 5, 6]            # durate dei lavori
k_cmax = 2                       # macchine identiche


def modello_cmax(d, k):
    n = len(d)
    m = nuovo_modello("makespan")
    x = m.addVars(n, k, vtype=GRB.BINARY, name="x")
    z = m.addVar(name="z")
    m.setObjective(z, GRB.MINIMIZE)
    m.addConstrs((x.sum(j, "*") == 1 for j in R(n)), name="lavoro")
    m.addConstrs((gp.quicksum(d[j] * x[j, mm] for j in R(n)) <= z for mm in R(k)),
                 name="carico")
    return m, x, z


m_cmax, x_cmax, z_var = modello_cmax(d_cmax, k_cmax)
z_cmax = risolvi(m_cmax)
salva_modello(m_cmax, "cap06_cmax")
print(f"  Durate {d_cmax} su {k_cmax} macchine identiche.")
print(f"  Il carico totale e' {sum(d_cmax)}: diviso per {k_cmax} da' "
      f"{frazione(sum(d_cmax) / k_cmax)}, e l'ottimo vale {frazione(z_cmax)}.")
for mm in R(k_cmax):
    lavori = [j + 1 for j in R(len(d_cmax)) if x_cmax[j, mm].X > 0.5]
    print(f"    macchina {mm + 1}: lavori {lavori}, carico "
          f"{sum(d_cmax[j - 1] for j in lavori)}")
assert z_cmax == sum(d_cmax) / k_cmax
salva_dati(pd.DataFrame([{"problema": "makespan", "z_milp": z_cmax}]), "cap06_cmax")
print("Fine.")

---

Notebook generato da `python/cap06_cmax.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE-CODE).